# 실습 1 — Tabular 분류 (13분 차시)

교재: [docs/01-tabular.md](../docs/01-tabular.md)

Titanic 데이터로 **로드 → EDA → 전처리 → 분할 → 모델 비교 → 튜닝 → 제출**을 한 바퀴 돕니다.
2~3분이면 끝납니다.

발표(45분)에서는 필요한 셀만 시연하고, 이 노트북은 **강의 후 각자 돌려 보는 자료**입니다.

> **실행 환경: Colab T4.** `런타임 → 런타임 유형 변경 → T4 GPU`로 먼저 바꾸고 시작하세요.


시험용 전체 파이프라인은 [templates/01_tabular_classification.py](../templates/01_tabular_classification.py)에 있습니다.
이 노트북은 **강의 중 따라 치는 축약본**입니다.#
> **참고 — 변수 이름:** 이 실습 노트북은 한 번에 하나만 돌리므로 접두사를 쓰지 않습니다.
> 시험 템플릿(`templates/`)은 한 커널에 여러 문제가 공존하므로 모든 변수에 `p1_`이 붙어 있습니다.
> 이유는 [docs/04-exam-day.md](../docs/04-exam-day.md)의 "변수 이름" 절에 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 습관 1: 시드 고정. 돌릴 때마다 점수가 바뀌면 개선 여부를 판단할 수 없다
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 0. 시험 환경 세팅

시험 시간은 **총 180분**입니다. 시작하면 세 가지를 먼저 합니다.
연습할 때부터 같은 순서로 하세요.

| | 언제 | 무엇을 |
|---|---|---|
| **세션당 한 번** | 시작 직후 | GPU 연결 확인, `PHONE`, 시드 |
| **문제마다** | 문제를 열 때 | `PROB_NO` 한 줄 |
| **계속** | 단계가 끝날 때마다 | 중간저장 — 단, 코드만 지켜집니다 |

GPU 확인은 세션 시작 때 한 번이면 됩니다(문제마다 다시 하지 않음). **튕기면 다시.**
실전 템플릿으로는 [templates/00_session_start.py](../templates/00_session_start.py)에 모여 있습니다.

In [ ]:
# --- 세션당 한 번 ---
PHONE = '01012345678'      # 본인 휴대폰 번호 (하이픈 없이)

# --- 문제마다 (시험 순서: Tabular = 1번) ---
PROB_NO = 1

SUBMIT_CSV = f'{PHONE}_{PROB_NO}.csv'
SUBMIT_MODEL = f'{PHONE}_{PROB_NO}.pkl'
print(f"제출 파일: {SUBMIT_CSV} / {SUBMIT_MODEL}")

## 1. 로드

시험에서는 `train.csv` / `test.csv`가 따로 주어집니다.
여기서는 Titanic을 8:2로 잘라 같은 상황을 만듭니다.

In [ ]:
df_all = sns.load_dataset('titanic')
TARGET = 'survived'

df_train, df_test = train_test_split(
    df_all, test_size=0.2, random_state=RANDOM_SEED, stratify=df_all[TARGET]
)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 습관 3: 클래스 수를 변수로. 이진이든 다중이든 아래 코드가 그대로 돈다
NUM_CLASSES = df_train[TARGET].nunique()

print(f"Train {df_train.shape} / Test {df_test.shape}")
print(f"클래스 {NUM_CLASSES}개")
df_train.head()

## 2. EDA — 그림 말고 표 하나

EDA의 목적은 감상이 아니라 **어느 컬럼을 버릴지 정하는 것**입니다.

In [ ]:
col_summary = pd.DataFrame({
    'dtype': df_train.dtypes,
    'missing': df_train.isnull().sum(),
    'miss_pct': (df_train.isnull().mean() * 100).round(1),
    'nunique': df_train.nunique(),
    'nunique_pct': (df_train.nunique() / len(df_train) * 100).round(1),
    'sample': df_train.iloc[0],
})
print(col_summary.to_string())

위 표를 기준표에 대입해 봅시다.

| 신호 | 조치 |
|---|---|
| `miss_pct` ≥ 70 | 제거 |
| `nunique_pct` ≥ 95 | 제거 (식별자) |
| `nunique` == 1 | 제거 |
| 타겟 상관 ≥ 0.95 | 제거 (누수) |
| 의미 중복 | 하나만 남김 |

`deck`은 결측이 77%로 기준을 넘습니다.
`pclass`와 `class`는 같은 정보이고, `embarked`와 `embark_town`도 마찬가지입니다.

그리고 가장 중요한 컬럼이 하나 숨어 있습니다. 다음 셀에서 찾아봅시다.

In [ ]:
# 타겟과의 관계를 보면 alive 가 survived 를 그대로 옮겨 놓은 컬럼임이 드러난다
print(pd.crosstab(df_train['alive'], df_train['survived']))

`alive`는 정답을 문자로 바꿔 놓은 것에 불과합니다. 이걸 남겨 두면 **정확도 100%**가 나오고,
그건 모델이 잘한 게 아니라 답안지를 보고 푼 것입니다.

> 시험에서 정확도가 0.99를 넘으면 기뻐하기 전에 누수를 의심하세요.

## 3. 전처리

순서: **제거 → 분리 → 결측 → 인코딩 → 스케일링**

In [ ]:
# 3-1. 제거 — train, test 에 똑같이!
drop_cols = ['alive', 'deck', 'class', 'embark_town', 'who', 'adult_male', 'alone']
df_train = df_train.drop(columns=drop_cols, errors='ignore')
df_test = df_test.drop(columns=drop_cols, errors='ignore')

# 3-2. X / y 분리
X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].copy()
X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET].copy()

# 3-3. 수치형 / 범주형 자동 분리 — 이후 코드가 데이터에 독립적이 된다
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"수치형 {num_cols}")
print(f"범주형 {cat_cols}")

In [ ]:
# 3-4. 결측치 — 습관 2: train 에서 fit, test 는 transform
num_imputer = SimpleImputer(strategy='median')          # 중앙값: 이상치에 강건
cat_imputer = SimpleImputer(strategy='most_frequent')

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])      # transform 만!

X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print(f"남은 결측 - train {X.isnull().sum().sum()} / test {X_test.isnull().sum().sum()}")

In [ ]:
# 3-5. 인코딩 — 트리 모델에는 LabelEncoder 로 충분
# train 에서 배운 값을 숫자 사전으로 만들고, test 는 사전으로 변환한다.
# test 에만 있는 값은 사전에 없으므로 NaN → -1 이 된다 (에러 대신 안전한 기본값)
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    code = {v: i for i, v in enumerate(le.classes_)}
    X_test[col] = X_test[col].astype(str).map(code).fillna(-1).astype(int)
    print(f"  {col}: {list(le.classes_)} -> {list(range(len(le.classes_)))}")

In [ ]:
# 3-6. 스케일링
# 트리 모델엔 불필요하지만, LogisticRegression 을 같은 표에서 비교하려면 필요하다
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
X_scaled.head()

### 잠깐 — 누수를 직접 확인해 봅시다

아래는 **하면 안 되는 코드**입니다. 왜 위험한지 눈으로 보기 위해서만 실행합니다.

In [ ]:
wrong_scaler = StandardScaler()
X_test_wrong = wrong_scaler.fit_transform(X_test)      # test 로 fit — 금지!

print("올바른 방식 (train 통계로 변환) - test 평균:")
print(X_test_scaled.mean().round(3).to_dict())
print("\n잘못된 방식 (test 자기 통계로 변환) - test 평균:")
print(pd.DataFrame(X_test_wrong, columns=X_test.columns).mean().round(3).to_dict())

잘못된 쪽은 평균이 정확히 0입니다. **깔끔해 보이는 게 함정입니다.**
test의 평균·분산 정보가 변환에 쓰였다는 뜻이고, 이건 실전에서 알 수 없는 정보입니다.

올바른 쪽은 0에서 살짝 벗어나 있습니다. 그게 정상입니다.

## 4. 분할

`X_test`는 제출용이라 여기 들어가지 않습니다. `train`만 다시 8:2로 나눕니다.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y   # stratify 필수
)
print(f"X_train {X_train.shape}  (학습)")
print(f"X_val   {X_val.shape}  (성능 측정)")
print(f"X_test  {X_test_scaled.shape}  (제출용 - 학습에도 평가에도 안 씀)")

## 5. 모델 비교

어떤 모델이 좋을지 이론으로 고민하지 말고 같은 조건에 넣고 돌립니다.
(XGBoost/LightGBM은 설치가 필요해 이 노트북에서는 뺐습니다. 템플릿에는 들어 있습니다.)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    results[name] = {
        'Accuracy': accuracy_score(y_val, y_pred),
        'F1_macro': f1_score(y_val, y_pred, average='macro'),
        'CV_Acc': cross_val_score(model, X_train, y_train, cv=5).mean(),
    }

results_df = pd.DataFrame(results).T.sort_values('Accuracy', ascending=False)
print(results_df.round(4))

best_model_name = results_df.index[0]
print(f"\n1위: {best_model_name}")

**`Accuracy`와 `CV_Acc`를 나란히 보세요.**

`Accuracy`는 val 한 번의 결과라 운이 섞입니다. `CV_Acc`는 5번 나눠 학습한 평균입니다.
둘이 크게 벌어지면 val 분할이 유리하게 걸린 것이니, CV가 높은 모델을 고르는 편이 안전합니다.

## 6. 튜닝 — 1위 하나만

5개를 전부 튜닝하면 시간이 남지 않습니다.

In [ ]:
# 비교에서 1위로 뽑힌 모델의 그리드만 꺼내 쓴다
param_grids = {
    'RandomForest': (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {'n_estimators': [100, 200, 300], 'max_depth': [5, 8, 12, None],
         'min_samples_split': [2, 5]},
    ),
    'GradientBoosting': (
        GradientBoostingClassifier(random_state=RANDOM_SEED),
        {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 8],
         'learning_rate': [0.05, 0.1]},
    ),
    'LogisticRegression': (
        LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
        {'C': [0.01, 0.1, 1, 10]},
    ),
}

est, param_grid = param_grids[best_model_name]
print(f"튜닝 대상: {best_model_name}")

grid = GridSearchCV(
    est, param_grid,
    cv=5,
    scoring='accuracy',      # 문제가 요구하는 지표로 바꿀 것
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print(f"최적 파라미터: {grid.best_params_}")
print(f"최적 CV 점수: {grid.best_score_:.4f}")

## 7. 평가

In [ ]:
best_model = grid.best_estimator_
y_val_pred = best_model.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred, average='macro')
print(f"Accuracy {acc:.4f} / F1(macro) {f1:.4f}")

grade = next((g for t, g in [(0.95, '매우 우수'), (0.90, '우수'),
                             (0.85, '양호'), (0.80, '보통')] if acc >= t), '미흡')
print(f"-> {grade}")
print()
print(classification_report(y_val, y_val_pred))

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# 특성 중요도 — 한 컬럼이 압도적이면 누수를 의심할 것
# (트리 계열에만 있는 속성. 선형 모델이 1위였다면 coef_ 를 보면 된다)
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(fi.round(4))
    fi.plot(kind='barh', figsize=(7, 4))
    plt.title('Feature Importance'); plt.tight_layout(); plt.show()

`alive`를 지우지 않았다면 이 그래프에서 그 컬럼 하나가 1.0에 가깝게 솟아 있었을 겁니다.
특성 중요도는 **누수 탐지기**로도 쓰입니다.

## 8. 제출

이 데이터는 타겟이 원래 0/1 정수라 복원할 게 없지만, 실전에서는 다릅니다.
**예측 컬럼명과 값이 요구된 label과 다르면 그 문제는 0점**입니다 —
타겟을 인코딩했다면 `inverse_transform`으로 되돌려서 내세요.

In [ ]:
test_predictions = best_model.predict(X_test_scaled)

submission = pd.DataFrame({
    'PassengerId': df_test.index,      # 실제 시험에서는 문제가 지정한 ID 컬럼
    'predicted': test_predictions,
})
submission.to_csv(SUBMIT_CSV, index=False)     # index=False 필수

import joblib
joblib.dump(best_model, SUBMIT_MODEL)          # 모델도 함께 제출

print(f"[Test 성능] Accuracy {accuracy_score(y_test, test_predictions):.4f}")
print(f"[저장] {SUBMIT_CSV} / {SUBMIT_MODEL}")
submission.head()

In [ ]:
# 제출 전 점검
import os
assert len(submission) == len(df_test), '행 수 불일치!'
assert submission.isnull().sum().sum() == 0, '결측 존재!'

# 파일이 실제로 만들어졌는지 눈으로 확인. to_csv 가 조용히 다른 경로에 쓰는 일이 있습니다.
made = [f for f in os.listdir('.') if f.startswith(PHONE)]
print('생성된 제출 파일:', made)
assert SUBMIT_CSV in made and SUBMIT_MODEL in made, '제출 파일 누락!'
print('점검 통과 - 행 수, 결측, 파일명 이상 없음')

# 여기까지 왔으면 Ctrl+S. 저장하지 않으면 세션이 끊길 때 전부 사라집니다.

---
## 혼자 해 볼 것

1. `drop_cols`에서 `'alive'`를 빼고 다시 돌려 보세요. 정확도와 특성 중요도가 어떻게 되나요?
2. `family_size = sibsp + parch + 1` 피처를 추가하면 점수가 오르나요? (train/test 둘 다에 적용할 것)
3. `stratify=y`를 빼고 분할하면 점수가 얼마나 흔들리나요?

다음: [실습 2 — Text](02_text_quickstart.ipynb) · 교재 [docs/02-text.md](../docs/02-text.md)